In [1]:
import os
import zipfile as zf
import argparse
import sys
import time

import numpy as np
import h5py
import librosa as lb
import torchaudio
import torch
import sounddevice as sd

import utils.config as config
import utils.utility_funcs as my_utils

In [2]:
PATH_TO_FSD50K_TRAIN_AUDIO = r"C:\Users\mp431591\Documents\FSD50K_dev_audio" # dev == train
PATH_TO_FSK50k_EVAL_AUDIO = r"C:\Users\mp431591\Documents\FSD50K_eval_audio"

PATH_TO_FSD50K_TRAIN_FNAMES_50 = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\fsd50k_train_filenames_top50.txt'
PATH_TO_FSD50K_EVAL_FNAMES_50 = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\fsd50k_eval_filenames_top50.txt'

PATH_TO_FSD50K_TRAIN_LABELS_50 = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\pickle_objects\fsd50k_eval_labels_50_dict.pckl'
PATH_TO_FSD50K_EVAL_LABELS_50 = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\pickle_objects\fsd50k_eval_labels_50_dict.pckl'

TARGET_SR = config.sample_rate


In [3]:
valid_filenames = set()
with open(PATH_TO_FSD50K_EVAL_FNAMES_50, 'r') as fnames:
    for line in fnames.readlines():
        valid_filenames.add(line.rstrip('\n'))

valid_filenames

{'415925',
 '131367',
 '137745',
 '80846',
 '100647',
 '371153',
 '400802',
 '361091',
 '90042',
 '162042',
 '203768',
 '75337',
 '69136',
 '266185',
 '58973',
 '59627',
 '77742',
 '391908',
 '220717',
 '367125',
 '44774',
 '430049',
 '58389',
 '344151',
 '187604',
 '37237',
 '277267',
 '257005',
 '371510',
 '338824',
 '52347',
 '124501',
 '381943',
 '155467',
 '127721',
 '321785',
 '17008',
 '88408',
 '118101',
 '391544',
 '98650',
 '351778',
 '343741',
 '57801',
 '56240',
 '28717',
 '329110',
 '349896',
 '414027',
 '149187',
 '388274',
 '86938',
 '58371',
 '389761',
 '171486',
 '81085',
 '64553',
 '326563',
 '22615',
 '91947',
 '148763',
 '351479',
 '367819',
 '256093',
 '371123',
 '9853',
 '429850',
 '128150',
 '57587',
 '70049',
 '381410',
 '406839',
 '31615',
 '406087',
 '242951',
 '212038',
 '346220',
 '189765',
 '415202',
 '391825',
 '72578',
 '48212',
 '403243',
 '139767',
 '72571',
 '362919',
 '369989',
 '404393',
 '196128',
 '109102',
 '118808',
 '197450',
 '172081',
 '392852

In [4]:
fsd50k_eval_labels = my_utils.pickle_load(PATH_TO_FSD50K_EVAL_LABELS_50)
print(fsd50k_eval_labels)

{'37199': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], '175151': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], '253463': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], '329838': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], '1277': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], '30149': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], '331398': [0, 0, 0, 0, 0, 0, 0,

In [5]:
MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(sample_rate=config.sample_rate,
                                                     n_fft=config.n_fft,
                                                     hop_length=config.hop_length,
                                                     n_mels=config.n_mels,
                                                     f_min=config.fmin,
                                                     f_max=config.fmax)

In [ ]:
counter = 0

start_time = time.time()
with os.scandir(PATH_TO_FSK50k_EVAL_AUDIO) as audio_dir:

    for idx, entry in enumerate(audio_dir):
        pure_fname = entry.name.rstrip('.wav')

        if pure_fname in valid_filenames:

            counter += 1
            audio, sr = torchaudio.load(entry.path)
            print(audio.shape[1]/sr)

            if sr != TARGET_SR:
                audio = torchaudio.functional.resample(audio, orig_freq=sr,
                                                        new_freq=TARGET_SR)
                print(f"Resampling file {pure_fname}")
            
            nr_of_audio_samples = audio.shape[1]
            duration = nr_of_audio_samples / TARGET_SR
            target_duration = 10
            split_chunks = []

            if duration > target_duration:
                split_chunks = my_utils.chunk_audio(audio, samplerate=TARGET_SR,
                                               target_duration=target_duration)
                # These are going to become their own separate datapoints
                for idx, chunk in enumerate(split_chunks):
                    fname_for_hdf5 = pure_fname + "_" + str(idx)
                    print(fname_for_hdf5)
            else:
                audio = my_utils.pad_or_truncate(audio, 
                                                 audio_length=(target_duration * TARGET_SR))
                mel_specgram = MEL_TRANSFORM(audio)
                mel_specgram = torch.transpose(mel_specgram, 2, 1)
                mel_specgram = torch.log(mel_specgram + torch.finfo(torch.float32).eps)
                
            print(split_chunks)
        if counter == 1:
            break
end_time = time.time()

print(f"Number of valid files: {counter}")
print(f"Finding {counter} valid files and processing them took {end_time-start_time} seconds")

18.553015873015873
Resampling file 100
100_0
100_1
[tensor([[0.0002, 0.0016, 0.0028,  ..., 0.0035, 0.0048, 0.0049]]), tensor([[ 0.0049,  0.0044,  0.0045,  ...,  0.0019,  0.0005, -0.0003]])]
Number of valid files: 1
Finding 1 valid files and processing them took 0.014689207077026367 seconds


Equal!
Equal!
